In [2]:
#========================a,THUẬT TOÁN AKT vào bài toán 8 puzzle(n=3)===========
import copy
from heapq import heappush, heappop

# Giả sử puzzle(n=3) n = 3
n = 3 # Explicitly define n
# 4 vị trí dịch chuyền tương ứng bottom, left, top, right
rows = [ 1, 0, -1, 0 ]
cols = [ 0, -1, 0, 1 ]

# Tạo một lớp hàng đợi
class priorityQueue:
  def __init__(self): # Corrected __init__
    self.heap = []
  # Inserting a new key 'key'
  def push(self, key):
      heappush (self.heap, key)
  #funct to remove the element that is min from the Priority Queue
  def pop(self):
      return heappop(self.heap)
  # funct to check if the Queue is empty or not
  def empty(self):
      if not self.heap:
          return True
      else:
          return False

# structure of the node
class nodes:
  def __init__(self, parent, mats, empty_tile_posi, costs, levels): # Corrected __init__
      self.parent = parent
      # Useful for Storing the matrix
      self.mats = mats
      self.empty_tile_posi = empty_tile_posi
      self.costs = costs
      self.levels = levels

  def __lt__ (self, nxt): # Corrected __lt__
      return self.costs < nxt.costs

def calculateCosts (mats, final) -> int:
  count = 0
  for i in range(n):
      for j in range(n):
          # Exclude empty tile (0) from cost calculation
          if ((mats[i][j] != 0) and (mats[i][j] != final[i][j])): count += 1
  return count

def newNodes (mats, empty_tile_posi, new_empty_tile_posi, levels, parent, final) -> nodes:
   # Copying data from the parent matrixes to the present matrixes
   new_mats = copy.deepcopy (mats)
   # Moving the tile by 1 position
   x1 = empty_tile_posi[0]
   y1 = empty_tile_posi[1]
   x2 = new_empty_tile_posi[0]
   y2 = new_empty_tile_posi[1]
   new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats [x1][y1]
   # Setting the no. of misplaced tiles
   costs = calculateCosts(new_mats, final)
   new_nodes = nodes(parent, new_mats, new_empty_tile_posi, costs, levels)
   return new_nodes

# func to print the N by N matrix
def printMatsrix(mats):
    for i in range(n):
        for j in range(n):
            print("%d " % (mats[i][j]), end = " ")
        print()

# func to know if (x, y) is a valid or invalid
# matrix coordinates
def isSafe(x, y):
    return x >= 0 and x < n and y >= 0 and y < n

# Printing the path from the root node to the final node
def printPath(root):
    if root == None:
        return
    printPath(root.parent)
    printMatsrix(root.mats)
    print()

def solve(initial, empty_tile_posi, final):
    pq = priorityQueue()
    # Creating the root node
    costs = calculateCosts (initial, final)
    root = nodes (None, initial, empty_tile_posi, costs, 0)
    # Adding root to the list of live nodes
    pq.push(root)

    while not pq.empty():
        minimum = pq.pop()
        if minimum.costs == 0:
            printPath(minimum)
            return

        # Generating all feasible children
        for i in range(4): # 4 possible moves (bottom, left, top, right)
            new_tile_posi = [
                minimum.empty_tile_posi[0] + rows[i],
                minimum.empty_tile_posi[1] + cols[i],
            ]
            if isSafe(new_tile_posi[0], new_tile_posi[1]):
                # Creating a child node
                child = newNodes(minimum.mats,
                                 minimum.empty_tile_posi,
                                 new_tile_posi,
                                 minimum.levels + 1,
                                 minimum, final,)
                # Adding the child to the list of live nodes
                pq.push(child)

# Initial configuration for 8-puzzle
initial = [ [ 1, 2, 3 ],
            [ 5, 6, 0 ],
            [ 7, 8, 4 ] ]
# Final configuration for 8-puzzle
final = [ [ 1, 2, 3 ],
          [ 5, 8, 6 ],
          [ 0, 7, 4 ] ]

# Position of the empty tile (0) in the initial state
empty_tile_posi = [1, 2]
solve(initial, empty_tile_posi, final)

#======================b, THUẬT TOÁN TÌM KIẾM A* TRÊN ĐỒ THỊ ===================
#b,Thuật toán tìm kiếm A* trên đồ thị
from collections import deque

class Graph:
  def __init__ (self, adjac_lis): # Corrected __init__
    self.adjac_lis = adjac_lis

  def get_neighbors(self, v):
      return self.adjac_lis[v]

  # This is heuristic function which is having equal values for all nodes
  def h(self, n):
    H = {
        'A': 1,
        'B': 1,
        'C': 1,
        'D': 1
    }
    return H[n]

  def a_star_algorithm(self, start, stop):
     open_1st = set([start])
     closed_1st = set([])

     poo = {} # Actual cost from start node to current node n
     poo[start] = 0

     # par contains an adjacency mapping of all nodes for path reconstruction
     par = {}
     par[start] = start

     while len(open_1st) > 0:
      n = None

      # It will find a node with the lowest value of f() = g(n) + h(n)
      for v in open_1st:
        if n == None or poo[v] + self.h(v) < poo[n] + self.h(n):
          n = v;

      if n == None:
        print('Path does not exist!')
        return None

      # If the current node is the stop node, reconstruct and return the path
      if n == stop:
        reconst_path = []

        while par[n] != n:
          reconst_path.append(n)
          n = par[n]

        reconst_path.append(start)
        reconst_path.reverse()

        print('Path found: {}'.format(reconst_path))
        return reconst_path

      # For all the neighbors of the current node
      for (m, weight) in self.get_neighbors(n):
        # If the current node is not present in both open_1st and closed_1st
        # add it to open_1st and note n as its par
        if m not in open_1st and m not in closed_1st:
          open_1st.add(m)
          par[m] = n
          poo[m] = poo[n] + weight
        else:
            # If it is in open_1st or closed_1st, check if new path is better

            if poo[m] > poo[n] + weight:
                poo[m] = poo[n] + weight
                par[m] = n

                if m in closed_1st:
                  closed_1st.remove(m)
                  open_1st.add(m)

      open_1st.remove(n)
      closed_1st.add(n)

     print('Path does not exist!')
     return None
adjac_lis = {
      'A':[('B', 1), ('C', 3), ('D', 7)],
      'B':[('C',1), ('D', 5)],
      'C': [('D',12)]
     }

graph1 = Graph(adjac_lis)
graph1.a_star_algorithm('A', 'C')

#----------------------bai tap tren lop------------------
#bai1:Cài đặt thuật giải AKT vào bài toán 15 puzzle(n=4).
import copy
from heapq import heappush, heappop

n = 4

rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class priorityQueue:
    def __init__(self):
        self.heap = []

    def push(self, key):
        heappush(self.heap, key)

    def pop(self):
        return heappop(self.heap)

    def empty(self):
        return len(self.heap) == 0


class nodes:
    def __init__(self, parent, mats, empty_tile_posi, costs, levels):
        self.parent = parent
        self.mats = mats
        self.empty_tile_posi = empty_tile_posi
        self.costs = costs
        self.levels = levels

    def __lt__(self, nxt):
        return (self.costs + self.levels) < (nxt.costs + nxt.levels)


def calculateCosts(mats, final):
    count = 0
    for i in range(n):
        for j in range(n):
            if mats[i][j] != 0 and mats[i][j] != final[i][j]:
                count += 1
    return count


def newNodes(mats, empty_tile_posi, new_empty_tile_posi, levels, parent, final):
    new_mats = copy.deepcopy(mats)

    x1, y1 = empty_tile_posi
    x2, y2 = new_empty_tile_posi

    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    costs = calculateCosts(new_mats, final)

    return nodes(parent, new_mats, new_empty_tile_posi, costs, levels)


def printMatrix(mats):
    for i in range(n):
        for j in range(n):
            print("%2d " % mats[i][j], end=" ")
        print()
    print()


def isSafe(x, y):
    return 0 <= x < n and 0 <= y < n


def printPath(root):
    if root is None:
        return
    printPath(root.parent)
    printMatrix(root.mats)


def solve(initial, empty_tile_posi, final):
    pq = priorityQueue()

    root = nodes(None, initial, empty_tile_posi,
                 calculateCosts(initial, final), 0)

    pq.push(root)

    visited = set()

    while not pq.empty():
        minimum = pq.pop()

        state_tuple = tuple(tuple(row) for row in minimum.mats)
        if state_tuple in visited:
            continue
        visited.add(state_tuple)

        if minimum.costs == 0:
            printPath(minimum)
            return

        for i in range(4):
            new_tile_posi = [
                minimum.empty_tile_posi[0] + rows[i],
                minimum.empty_tile_posi[1] + cols[i],
            ]

            if isSafe(new_tile_posi[0], new_tile_posi[1]):
                child = newNodes(minimum.mats,
                                 minimum.empty_tile_posi,
                                 new_tile_posi,
                                 minimum.levels + 1,
                                 minimum,
                                 final)
                pq.push(child)


initial = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 0],
    [13, 14, 15, 12]
]

final = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 0]
]

empty_tile_posi = [2, 3]

solve(initial, empty_tile_posi, final)
#======bai2:Cài đặt thuật giải A* tìm đường đi giữa 2 đỉnh bất kỳ trong đồ thị.
from collections import deque

class Graph:
    def __init__(self, adjac_lis, heuristic):
        self.adjac_lis = adjac_lis
        self.heuristic = heuristic  # Dictionary of heuristics for each node

    def get_neighbors(self, v):
        """Trả về các đỉnh láng giềng của v cùng với trọng số"""
        return self.adjac_lis[v]

    def h(self, n):
        """Hàm heuristic, trả về giá trị heuristic cho đỉnh n"""
        return self.heuristic.get(n, float('inf'))  # Nếu không có heuristic thì trả về vô cùng

    def a_star_algorithm(self, start, stop):
        # Mở và đóng danh sách
        open_1st = set([start])  # Mở danh sách
        closed_1st = set([])     # Danh sách đã thăm

        poo = {}  # chi phí thực tế từ start đến node n
        poo[start] = 0

        par = {}  # Chứa cây cấu trúc đồ thị (để tạo lại đường đi)
        par[start] = start

        while len(open_1st) > 0:
            n = None

            # Tìm đỉnh có giá trị f(n) = g(n) + h(n) thấp nhất
            for v in open_1st:
                if n is None or poo[v] + self.h(v) < poo[n] + self.h(n):
                    n = v

            if n is None:
                print('Path does not exist!')
                return None

            # Nếu node hiện tại là đích, xây dựng lại đường đi
            if n == stop:
                reconst_path = []
                while par[n] != n:
                    reconst_path.append(n)
                    n = par[n]
                reconst_path.append(start)
                reconst_path.reverse()
                print('Path found: {}'.format(reconst_path))
                return reconst_path

            # Xử lý các láng giềng của node hiện tại
            for (m, weight) in self.get_neighbors(n):
                if m not in open_1st and m not in closed_1st:
                    open_1st.add(m)
                    par[m] = n
                    poo[m] = poo[n] + weight
                else:
                    # Nếu đỉnh đã có trong open_1st hoặc closed_1st, kiểm tra xem có cần cập nhật không
                    if poo[m] > poo[n] + weight:
                        poo[m] = poo[n] + weight
                        par[m] = n

                        if m in closed_1st:
                            closed_1st.remove(m)
                            open_1st.add(m)

            open_1st.remove(n)
            closed_1st.add(n)

        print('Path does not exist!')
        return None

# Định nghĩa đồ thị dưới dạng danh sách kề (adjacency list)
adjac_lis = {
    'A': [('B', 1), ('C', 3), ('D', 7)],
    'B': [('C', 1), ('D', 5)],
    'C': [('D', 12)],
    'D': []  # D không có láng giềng
}

# Hàm heuristic (ước lượng khoảng cách còn lại)
from collections import deque

class Graph:
    def __init__(self, adjac_lis, heuristic):
        self.adjac_lis = adjac_lis
        self.heuristic = heuristic  # Dictionary of heuristics for each node

    def get_neighbors(self, v):
        """Trả về các đỉnh láng giềng của v cùng với trọng số"""
        return self.adjac_lis[v]

    def h(self, n):
        """Hàm heuristic, trả về giá trị heuristic cho đỉnh n"""
        return self.heuristic.get(n, float('inf'))  # Nếu không có heuristic thì trả về vô cùng

    def a_star_algorithm(self, start, stop):
        # Mở và đóng danh sách
        open_1st = set([start])  # Mở danh sách
        closed_1st = set([])     # Danh sách đã thăm

        poo = {}  # chi phí thực tế từ start đến node n
        poo[start] = 0

        par = {}  # Chứa cây cấu trúc đồ thị (để tạo lại đường đi)
        par[start] = start

        while len(open_1st) > 0:
            n = None

            # Tìm đỉnh có giá trị f(n) = g(n) + h(n) thấp nhất
            for v in open_1st:
                if n is None or poo[v] + self.h(v) < poo[n] + self.h(n):
                    n = v

            if n is None:
                print('Path does not exist!')
                return None

            # Nếu node hiện tại là đích, xây dựng lại đường đi
            if n == stop:
                reconst_path = []
                while par[n] != n:
                    reconst_path.append(n)
                    n = par[n]
                reconst_path.append(start)
                reconst_path.reverse()
                print('Path found: {}'.format(reconst_path))
                return reconst_path

            # Xử lý các láng giềng của node hiện tại
            for (m, weight) in self.get_neighbors(n):
                if m not in open_1st and m not in closed_1st:
                    open_1st.add(m)
                    par[m] = n
                    poo[m] = poo[n] + weight
                else:
   # Nếu đỉnh đã có trong open_1st hoặc closed_1st, kiểm tra xem có cần cập nhật không
                    if poo[m] > poo[n] + weight:
                        poo[m] = poo[n] + weight
                        par[m] = n

                        if m in closed_1st:
                            closed_1st.remove(m)
                            open_1st.add(m)

            open_1st.remove(n)
            closed_1st.add(n)

        print('Path does not exist!')
        return None

# Định nghĩa đồ thị dưới dạng danh sách kề (adjacency list)
adjac_lis = {
    'A': [('B', 1), ('C', 3), ('D', 7)],
    'B': [('C', 1), ('D', 5)],
    'C': [('D', 12)],
    'D': []  # D không có láng giềng
}

# Hàm heuristic (ước lượng khoảng cách còn lại)
# Ví dụ: A, B, C, D có heuristic như sau:
heuristic = {
    'A': 4,
    'B': 2,
    'C': 1,
    'D': 0
}

# Tạo đối tượng Graph
graph1 = Graph(adjac_lis, heuristic)

# Tìm đường đi từ A đến C
graph1.a_star_algorithm('A', 'C')


1  2  3  
5  6  0  
7  8  4  

1  2  3  
5  0  6  
7  8  4  

1  2  3  
5  8  6  
7  0  4  

1  2  3  
5  8  6  
0  7  4  

Path found: ['A', 'B', 'C']
 1   2   3   4  
 5   6   7   8  
 9  10  11   0  
13  14  15  12  

 1   2   3   4  
 5   6   7   8  
 9  10  11  12  
13  14  15   0  

Path found: ['A', 'B', 'C']


['A', 'B', 'C']